#  Projekt 7 - Optymalizacja modeli Ultralytics YOLO

Poniższy skrypt należy uruchomić w środowisku Google Colab.

## Model

*YOLO26n* jest najmniejszym modelem z rodziny YOLO26 (*You-Only-Look-Once*). Jest to model real-time detekcji obiektów. 

In [ ]:
!pip install ultralytics onnx onnxruntime onnxruntime-tools

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")

## Pomiar czasu wnioskowania

Wykorzystując poniższy skrypt możemy w prosty sposób zmierzyć czas wnioskowania modelu.

In [ ]:
import urllib.request
image_path = "https://github.com/ultralytics/assets/releases/download/v0.0.0/bus.jpg"
urllib.request.urlretrieve(image_path, "bus.jpg")


In [ ]:
import time
import cv2 as cv, numpy as np

def inference(model: YOLO, image: np.ndarray): model(image, verbose=False, device="cpu", imgsz=640)

def measure_inference_time(model: YOLO, image_path: str, n_iterations: int=100):

    # Load image
    image = cv.imread(image_path)

    # Coldstart - pierwsze wnioskowanie, zazwyczaj wolniejsze ze względu na inicjalizację modelu i wczytanie wag
    start = time.time()
    inference(model,image)
    coldstart_time = time.time() - start
    
    # Pomiar dla N iteracji
    times = []
    for _ in range(n_iterations):
        start = time.time()
        inference(model,image)
        times.append(time.time() - start)
    
    avg_time = sum(times) / len(times)
    
    return {
        "average_inference_time": avg_time * 1000,
        "min_time": min(times) * 1000,
        "max_time": max(times) * 1000,
    }

In [ ]:
timing_results = measure_inference_time(model, "bus.jpg", n_iterations=10)
print(f"Average inference time: {timing_results['average_inference_time']:.1f}ms")
print(f"Min time: {timing_results['min_time']:.1f}ms")
print(f"Max time: {timing_results['max_time']:.1f}ms")


Czy istnieją sposoby na zmniejszenie czasu wnioskowania (poprawę wydajności)?

### Zapis do formatu ONNX

Format *ONNX* działa jak uniwersalny tłumacz w świecie sztucznej inteligencji, który pozwala uniezależnić modele od konkretnego środowiska programistycznego. Zamiast zmuszać system docelowy do instalowania ciężkich bibliotek takich jak PyTorch czy TensorFlow, model jest eksportowany do otwartego standardu opartego na uniwersalnym grafie obliczeniowym. Dzięki temu ten sam plik możesz bezproblemowo uruchomić na procesorach Intel, AMD, architekturze ARM (np. Raspberry Pi) czy układach graficznych NVIDIA, używając do tego lekkiego i zoptymalizowanego silnika ONNX Runtime.

### Kwantyzacja

Kwantyzacja *INT8* to z kolei proces drastycznej optymalizacji matematycznej, który polega na zamianie 32-bitowych wag zmiennoprzecinkowych (FP32) na 8-bitowe liczby całkowite (INT8). W praktyce oznacza to "zaokrąglenie" skomplikowanych ułamków do prostych wartości z zakresu od -128 do 127, co natychmiast zmniejsza rozmiar pliku modelu aż o 75% i drastycznie redukuje zużycie pamięci RAM. Ponieważ sprzęt komputerowy przetwarza operacje na liczbach całkowitych o wiele szybciej i przy mniejszym poborze energii, model zyskuje potężny przyrost klatek na sekundę (FPS) przy jednoczesnym zachowaniu niemal identycznej dokładności detekcji.

In [ ]:
model.export(
    format="onnx",
    data="coco8.yaml",
    quantize=8,
) # ONNX + INT8

In [ ]:
model_onnx_int8 = YOLO("yolo26n_int8.onnx", task="detect")

timing_results = measure_inference_time(model_onnx_int8, "bus.jpg", n_iterations=10)
print(f"Average inference time: {timing_results['average_inference_time']:.1f}ms")
print(f"Min time: {timing_results['min_time']:.1f}ms")
print(f"Max time: {timing_results['max_time']:.1f}ms")

Na nowoczesnym PC możemy zaobserwować pogorszenie wydajności. Jaki będzie zysk czasowy na mikro-komputerze Raspberry Pi?